# 11.1 交付与预测

用登记好的模型文件出下个月的预测，并按一条明确的规则标出"需要提前备货"的 SKU。
这一格之后就是日常使用的入口：换个月份重跑，就是下个月的清单。

In [1]:
# 参数：dsflow run 时用 --param MONTH=2026-08 换月份
MONTH = "2026-07"
RATIO = 1.3
MIN_QTY = 10


In [2]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import add_history, add_season, out, predict

OUT = out("11.1")
PANEL = out("3.1") / "panel.parquet"
MODEL_PATH = out("7.1") / "model.json"
run = dsflow.start_run("11.1", project=ROOT, hypothesis=f"用登记的模型出 {MONTH} 各 SKU 需求量预测，并标出需要提前备货的 SKU")

model = json.loads(MODEL_PATH.read_text(encoding="utf-8"))
run.log_input(PANEL, name="sku_month_panel")
panel = pd.read_parquet(PANEL)
print(f"模型：{model['type']}，特征 {model['features']}，拟合数据 {model['trained_on']}")
print(f"面板最后一个月 {panel['月份'].max()}，本次预测 {MONTH}")


模型：线性回归（最小二乘，预测值截到 0 以上），特征 ['lag1', 'lag2', 'lag3', 'mean6', 'season']，拟合数据 2025-01 至 2026-03（训练 + 验证期）
面板最后一个月 2026-06，本次预测 2026-07


In [3]:
nxt = panel.drop_duplicates("SKU")[["SKU", "类目"]].assign(月份=MONTH, 需求量=np.nan, 划分="预测")
full = add_history(pd.concat([panel, nxt], ignore_index=True))
target = add_season(full[full["月份"] == MONTH], model["season"])
target["预测需求量"] = np.round(predict(model, target), 2)
target["提前备货"] = (target["预测需求量"] >= RATIO * target["mean3"]) & (target["预测需求量"] >= MIN_QTY)
result = (target[["SKU", "类目", "预测需求量", "mean3", "提前备货"]].rename(columns={"mean3": "近3月均值"})
          .sort_values(["预测需求量", "SKU"], ascending=[False, True]))
file = OUT / f"forecast_{MONTH}.csv"
result.to_csv(file, index=False)
result.head(10)


,SKU,类目,预测需求量,近3月均值,提前备货
10499,SKU00420,办公通用物资,20.18,24.000000,False
19649,SKU00786,办公通用物资,17.21,45.000000,False
5649,SKU00226,办公通用物资,16.37,43.666667,False
23499,SKU00940,办公通用物资,16.19,17.666667,False
5349,SKU00214,办公通用物资,15.71,36.000000,False
19074,SKU00763,办公通用物资,15.46,28.333333,False
39949,SKU01598,MRO工业品,15.27,18.666667,False
7549,SKU00302,办公通用物资,15.07,9.666667,True
749,SKU00030,办公通用物资,14.66,17.333333,False
11749,SKU00470,办公通用物资,14.40,16.666667,False


In [4]:
run.log_params({"模型文件": MODEL_PATH.resolve().relative_to(ROOT).as_posix(),
                "模型版本": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()[:12],
                "预测月份": MONTH, "提前备货规则": f"预测需求量 ≥ 近 3 月均值 × {RATIO} 且 ≥ {MIN_QTY}"})
metrics = {"预测SKU数": len(result), "提前备货SKU数": int(result["提前备货"].sum()),
           "预测需求量合计": round(float(result["预测需求量"].sum()), 2)}
run.log_metrics(metrics)
run.log_artifact(file, purpose=f"{MONTH} 各 SKU 需求量预测与提前备货标记", kind="table")
print(json.dumps(metrics, ensure_ascii=False, indent=1))
print(result[result["提前备货"]].head(5).to_string(index=False))


{
 "预测SKU数": 3000,
 "提前备货SKU数": 42,
 "预测需求量合计": 23077.96
}
     SKU     类目  预测需求量     近3月均值  提前备货
SKU00302 办公通用物资  15.07  9.666667  True
SKU02758   电力物资  13.96 10.666667  True
SKU01308 MRO工业品  13.52  9.000000  True
SKU02321   电力物资  13.12  2.666667  True
SKU00797 办公通用物资  12.88  4.333333  True


In [5]:
conclusion = (f"{MONTH} 预测 {metrics['预测SKU数']:,} 个 SKU，需求量合计 {metrics['预测需求量合计']:,}；"
              f"提示提前备货 {metrics['提前备货SKU数']} 个")
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


2026-07 预测 3,000 个 SKU，需求量合计 23,077.96；提示提前备货 42 个
